# Simulated right-belt speeds

Real-treadmill right-belt speed magnitudes for P1, P9, P12, and P15 at 1.0, 1.4, and 1.8 m/s. Colors use the same five-speed `rocket_r` mapping as the joint-angle figures.

In [5]:
# Configuration
RESULT_SUBFOLDER = "ideal_0814_e300_grf10_real_0815_e300_grf10"  # "grf3_angles1"
PARTICIPANTS = ["P1", "P9", "P12", "P14"]
PLOT_SPEEDS = [1.0, 1.4, 1.8]
ALL_SPEEDS = [1.0, 1.2, 1.4, 1.6, 1.8]
EXCLUDE_OBJECTIVE_OUTLIERS = True
EXPORT_FIGURE = True

In [3]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.lines import Line2D

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / "result_HC").is_dir() else cwd.parent
safe_name = re.sub(r"[^A-Za-z0-9]+", "_", RESULT_SUBFOLDER).strip("_")
trajectory_path = ROOT / "result_HC" / f"simulation_results_101_points_{safe_name}.csv"
outlier_path = ROOT / "result_HC" / f"objective_outliers_{safe_name}.csv"

df = pd.read_csv(trajectory_path)
if EXCLUDE_OBJECTIVE_OUTLIERS:
    outlier_rows = pd.read_csv(outlier_path)
    exclusion_column = (
        "exclude_pair" if "exclude_pair" in outlier_rows.columns
        else "objective_outlier"
    )
    if exclusion_column in outlier_rows.columns:
        is_outlier = (
            outlier_rows[exclusion_column].eq(True)
            | outlier_rows[exclusion_column].astype(str).str.lower().eq("true")
        )
        outlier_rows = outlier_rows.loc[is_outlier]
    excluded_trials = outlier_rows[["participant", "speed"]].drop_duplicates()
    trial_index = pd.MultiIndex.from_frame(df[["participant", "speed"]])
    excluded_index = pd.MultiIndex.from_frame(excluded_trials)
    df = df.loc[~trial_index.isin(excluded_index)].copy()

belt_data = df[
    df["participant"].isin(PARTICIPANTS)
    & df["speed"].isin(PLOT_SPEEDS)
    & df["treadmill_model"].eq("real")
].copy()
belt_data[["participant", "speed", "treadmill_model"]].drop_duplicates()

FileNotFoundError: [Errno 2] No such file or directory: '/home/rzlin/ys64ofuj/ideal-treadmill-biosym/result_HC/simulation_results_101_points_0804_1251.csv'

In [6]:
speed_colors = dict(zip(ALL_SPEEDS, sns.color_palette("rocket_r", len(ALL_SPEEDS))))
sns.set_theme(style="ticks")
fig, axes = plt.subplots(
    1, 4, figsize=(14, 3), sharex=True, sharey=True, constrained_layout=True,
)

for ax, participant in zip(axes, PARTICIPANTS):
    participant_data = belt_data[belt_data["participant"] == participant]
    for speed in PLOT_SPEEDS:
        trajectory = participant_data[participant_data["speed"] == speed].sort_values("node")
        ax.plot(
            trajectory["percent_gait"], trajectory["belt_speed_r_abs"]-speed,
            color=speed_colors[speed], linewidth=2.2,
        )
    ax.set_title(participant, fontsize=14)
    ax.set_xlabel("Gait cycle (%)", fontsize=14)
    ax.set_xlim(0, 100)

axes[0].set_ylabel("simulated belt \n speed change[m/s]", fontsize=14)
legend_handles = [
    Line2D([0], [0], color=speed_colors[speed], linewidth=3, label=f"{speed:.1f} m/s")
    for speed in PLOT_SPEEDS
]
fig.legend(
    handles=legend_handles, title="Nominal speed",
    loc="outside right center", frameon=False, fontsize=14, title_fontsize=14
)
#fig.suptitle("simulated belt speed change[m/s]")
sns.despine(fig=fig)

if EXPORT_FIGURE:
    output_path = ROOT / "result_HC" / f"right_belt_speeds_{safe_name}.pdf"
    fig.savefig(output_path, bbox_inches="tight")
    print(f"Saved {output_path.relative_to(ROOT)}")

plt.show()

NameError: name 'sns' is not defined